In [ ]:
# OPTIONAL-INSTALL REQUIRED PACKAGES-IF NEEDED
# ============================================================
# INSTALL ALL REQUIRED PACKAGES
# ============================================================

import sys
import subprocess

packages = [
    "segmentation-models-pytorch",
    "albumentations",
    "opencv-python-headless",
    "pandas",
    "numpy",
    "matplotlib",
    "timm",
]

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-U",
    *packages,
])

print("\nAll required packages were installed successfully.")
print("Restart the Jupyter kernel before running the training code.")

In [1]:
# MODEL TRAINING CODE
# ============================================================
# IMPORTS
# ============================================================

from pathlib import Path

import cv2
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import (
    Dataset,
    DataLoader,
    ConcatDataset
)

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp

import json
import matplotlib.pyplot as plt

from PIL import Image

import json
import matplotlib.pyplot as plt

from PIL import Image

import random

# ============================================================
# PROJECT AND DATASET PATHS
# ============================================================

PROJECT_ROOT = Path(".")

DATASET_ROOT = PROJECT_ROOT

TRAIN_DIR = DATASET_ROOT / "train"
VALID_DIR = DATASET_ROOT / "valid"
TEST_DIR = DATASET_ROOT / "test"


# ============================================================
# OUTPUT PATHS
# ============================================================

OUTPUT_DIR = PROJECT_ROOT / "deeplab_results"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BEST_MODEL_PATH = (
    OUTPUT_DIR / "best_deeplabv3plus.pt"
)

LAST_MODEL_PATH = (
    OUTPUT_DIR / "last_deeplabv3plus.pt"
)

HISTORY_PATH = (
    OUTPUT_DIR / "training_history.csv"
)


# ============================================================
# TRAINING PARAMETERS
# ============================================================

IMG_SIZE = 1024
BATCH_SIZE = 8
EPOCHS = 50

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

# Early stopping settings
PATIENCE = 15
MIN_DELTA = 5e-4

ENCODER = "resnet50"
ENCODER_WEIGHTS = "imagenet"

# Zero is safest for HPC Jupyter notebooks.
NUM_WORKERS = 0

# ============================================================
# RANDOM SEED
# ============================================================

SEED = 67

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Random seed set to {SEED}")

# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

AMP_ENABLED = DEVICE.type == "cuda"

print("Project root:", PROJECT_ROOT.resolve())
print("Dataset root:", DATASET_ROOT.resolve())
print("Output folder:", OUTPUT_DIR.resolve())
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# READ ROBOFLOW CLASS MAPPING
# ============================================================

def load_class_map(train_directory):

    csv_path = Path(train_directory) / "_classes.csv"

    if not csv_path.is_file():

        raise FileNotFoundError(
            f"Could not find class file:\n"
            f"{csv_path.resolve()}"
        )

    dataframe = pd.read_csv(csv_path)

    dataframe.columns = [
        str(column).strip()
        for column in dataframe.columns
    ]

    required_columns = {
        "Pixel Value",
        "Class"
    }

    if not required_columns.issubset(
        dataframe.columns
    ):

        raise ValueError(
            "The _classes.csv file must contain:\n"
            "'Pixel Value' and 'Class'\n\n"
            f"Columns found: "
            f"{dataframe.columns.tolist()}"
        )

    pixel_to_class = {}

    for class_index, row in dataframe.iterrows():

        pixel_value = int(
            row["Pixel Value"]
        )

        pixel_to_class[pixel_value] = int(
            class_index
        )

    class_names = (
        dataframe["Class"]
        .astype(str)
        .str.strip()
        .tolist()
    )

    return (
        pixel_to_class,
        class_names,
        dataframe
    )


pixel_to_class, class_names, class_dataframe = (
    load_class_map(TRAIN_DIR)
)

NUM_CLASSES = len(class_names)

display(class_dataframe)

print("Class names:", class_names)
print("Number of classes:", NUM_CLASSES)
print("Pixel mapping:", pixel_to_class)


# ============================================================
# SEMANTIC SEGMENTATION DATASET
# ============================================================

class SemanticSegmentationDataset(Dataset):

    def __init__(
        self,
        split_directory,
        pixel_to_class,
        transform=None
    ):

        self.split_directory = Path(
            split_directory
        )

        self.pixel_to_class = (
            pixel_to_class
        )

        self.transform = transform

        self.samples = []

        supported_extensions = {
            ".jpg",
            ".jpeg",
            ".png"
        }

        for image_path in sorted(
            self.split_directory.iterdir()
        ):

            if (
                image_path.suffix.lower()
                not in supported_extensions
            ):
                continue

            # Exclude semantic mask files.
            if image_path.stem.lower().endswith(
                "_mask"
            ):
                continue

            mask_path = (
                self.split_directory
                / f"{image_path.stem}_mask.png"
            )

            if mask_path.is_file():

                self.samples.append(
                    (
                        image_path,
                        mask_path
                    )
                )

        if len(self.samples) == 0:

            raise RuntimeError(
                "No image-mask pairs were found in:\n"
                f"{self.split_directory.resolve()}\n\n"
                "Expected filenames such as:\n"
                "example.jpg\n"
                "example_mask.png"
            )

        print(
            f"{self.split_directory.name}: "
            f"{len(self.samples)} image-mask pairs"
        )

    def __len__(self):

        return len(self.samples)

    def __getitem__(self, index):

        image_path, mask_path = (
            self.samples[index]
        )

        # -------------------------------
        # Load RGB image
        # -------------------------------

        image = cv2.imread(
            str(image_path),
            cv2.IMREAD_COLOR
        )

        if image is None:

            raise RuntimeError(
                f"Could not read image:\n"
                f"{image_path}"
            )

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        # -------------------------------
        # Load semantic PNG mask
        # -------------------------------

        raw_mask = cv2.imread(
            str(mask_path),
            cv2.IMREAD_GRAYSCALE
        )

        if raw_mask is None:

            raise RuntimeError(
                f"Could not read mask:\n"
                f"{mask_path}"
            )

        # -------------------------------
        # Convert Roboflow pixel values
        # to consecutive class indices
        # -------------------------------

        mask = np.zeros(
            raw_mask.shape,
            dtype=np.int64
        )

        for pixel_value, class_index in (
            self.pixel_to_class.items()
        ):

            mask[
                raw_mask == int(pixel_value)
            ] = int(class_index)

        # -------------------------------
        # Resize, normalize and convert
        # to PyTorch tensors
        # -------------------------------

        if self.transform is not None:

            transformed = self.transform(
                image=image,
                mask=mask
            )

            image = transformed["image"]
            mask = transformed["mask"]

        return (
            image.float(),
            mask.long()
        )

# -------------------------
# Synthetic Dataset
# -------------------------

SYN_DIR = PROJECT_ROOT / "syn_images"

# -------------------------
# Synthetic Augmented
# -------------------------

SYNAUG_DIR = PROJECT_ROOT / "synaug_train_5x"

# ============================================================
# SYNTHETIC DATASET
#
# syn_images/
# synaug_train_10x/
#
# Format:
#
# xxx_synthesized_image.jpg
# xxx_input_label.jpg
#
# RGB LABELS
# ============================================================

class PalmOilAugmentedSyntheticDataset(
    Dataset
):

    def __init__(
        self,
        images_dir,
        masks_dir
    ):

        self.images_dir = Path(
            images_dir
        )

        self.masks_dir = Path(
            masks_dir
        )

        self.filenames = [

            file.name

            for file in sorted(
                self.images_dir.iterdir()
            )

            if (
                file.is_file()

                and

                "synthesized_image.jpg"
                in file.name
            )
        ]

        print(

            f"{self.images_dir.name}: "

            f"{len(self.filenames)} "

            "synthetic samples"

        )

    def __len__(self):

        return len(
            self.filenames
        )

    def rgb_mask_to_class_indices(
        self,
        mask_pil
    ):

        mask_np = np.array(
            mask_pil
        )

        height, width, _ = (
            mask_np.shape
        )

        label_mask = np.zeros(

            (
                height,
                width
            ),

            dtype=np.int64

        )

        # ---------------------------------

        # CLASS 1

        # Ground Weeds

        # ---------------------------------

        weeds_condition = (

            (mask_np[:, :, 0] < 50)

            &

            (mask_np[:, :, 1] > 200)

            &

            (mask_np[:, :, 2] < 50)

        )

        label_mask[
            weeds_condition
        ] = 1

        # ---------------------------------

        # CLASS 2

        # Trunk

        # ---------------------------------

        trunks_condition = (

            (mask_np[:, :, 0] > 120)

            &

            (mask_np[:, :, 1] > 50)

            &

            (mask_np[:, :, 2] < 40)

        )

        label_mask[
            trunks_condition
        ] = 2

        return torch.from_numpy(
            label_mask
        )

    def __getitem__(

        self,

        index

    ):

        image_name = (
            self.filenames[index]
        )

        base_id = image_name.split(
            "_synthesized_image.jpg"
        )[0]

        mask_name = (
            f"{base_id}"
            "_input_label.jpg"
        )

        image_path = (
            self.images_dir
            /
            image_name
        )

        mask_path = (
            self.masks_dir
            /
            mask_name
        )

        image = Image.open(
            image_path
        ).convert(
            "RGB"
        )

        mask = Image.open(
            mask_path
        ).convert(
            "RGB"
        )

        # Convert PIL images to NumPy arrays
        image = np.array(image)
        mask = np.array(mask)
        
        # Convert RGB mask to class indices
        mask = self.rgb_mask_to_class_indices(
            Image.fromarray(mask)
        ).numpy()
        
        # Apply the SAME Albumentations transform as the real dataset
        if train_transform is not None:
            transformed = train_transform(
                image=image,
                mask=mask
            )
            
            image = transformed["image"]
            mask = transformed["mask"]
            
        return (
            image.float(),
            mask.long()
        )

# ============================================================
# TRANSFORMS
# ============================================================

train_transform = A.Compose([
    A.Resize(
        IMG_SIZE,
        IMG_SIZE
    ),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2()
])

valid_transform = A.Compose([
    A.Resize(
        IMG_SIZE,
        IMG_SIZE
    ),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2()
])


# ============================================================
# REAL DATASETS
# ============================================================

real_train_dataset = SemanticSegmentationDataset(
    split_directory=TRAIN_DIR,
    pixel_to_class=pixel_to_class,
    transform=train_transform
)

valid_dataset = SemanticSegmentationDataset(
    split_directory=VALID_DIR,
    pixel_to_class=pixel_to_class,
    transform=valid_transform
)

test_dataset = SemanticSegmentationDataset(
    split_directory=TEST_DIR,
    pixel_to_class=pixel_to_class,
    transform=valid_transform
)


# ============================================================
# SYNTHETIC DATASETS
# ============================================================

synthetic_dataset = PalmOilAugmentedSyntheticDataset(
    images_dir=SYN_DIR,
    masks_dir=SYN_DIR
)

synthetic_aug_dataset = PalmOilAugmentedSyntheticDataset(
    images_dir=SYNAUG_DIR,
    masks_dir=SYNAUG_DIR
)


# ============================================================
# COMBINE TRAINING DATA
# ============================================================

combined_train_dataset = ConcatDataset([
    real_train_dataset,
    synthetic_dataset,
    synthetic_aug_dataset
])

print()

print("========== Dataset Summary ==========")

print(f"Real Train          : {len(real_train_dataset)}")
print(f"Synthetic           : {len(synthetic_dataset)}")
print(f"Synthetic Augmented : {len(synthetic_aug_dataset)}")
print(f"Combined Train      : {len(combined_train_dataset)}")
print(f"Validation          : {len(valid_dataset)}")
print(f"Test                : {len(test_dataset)}")


# ============================================================
# DATALOADERS
# ============================================================

train_loader = DataLoader(
    combined_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)


print()

print("========== DataLoaders ==========")

print(f"Training batches   : {len(train_loader)}")
print(f"Validation batches : {len(valid_loader)}")
print(f"Test batches       : {len(test_loader)}")

# ============================================================
# DEEPLABV3+ MODEL
# ============================================================

model = smp.DeepLabV3Plus(
    encoder_name=ENCODER,
    encoder_weights=ENCODER_WEIGHTS,
    in_channels=3,
    classes=NUM_CLASSES,
    activation=None
)

model = model.to(DEVICE)


# ============================================================
# LOSS FUNCTIONS
# ============================================================

dice_loss = smp.losses.DiceLoss(
    mode="multiclass",
    from_logits=True
)

cross_entropy_loss = (
    nn.CrossEntropyLoss()
)


def combined_loss(
    predictions,
    targets
):

    dice = dice_loss(
        predictions,
        targets
    )

    cross_entropy = (
        cross_entropy_loss(
            predictions,
            targets
        )
    )

    return dice + cross_entropy


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


# ============================================================
# POLYNOMIAL LEARNING-RATE SCHEDULER
# ============================================================

def polynomial_decay(epoch):

    progress = min(
        epoch / EPOCHS,
        1.0
    )

    return (
        1.0 - progress
    ) ** 0.9


scheduler = (
    torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=polynomial_decay
    )
)


# ============================================================
# AUTOMATIC MIXED PRECISION
# ============================================================

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=AMP_ENABLED
)

print("DeepLabV3+ model created")
print("Encoder:", ENCODER)
print("Number of output classes:", NUM_CLASSES)
print(
    "Model device:",
    next(model.parameters()).device
)


# ============================================================
# METRICS
# ============================================================

class SegmentationMetrics:

    def __init__(self, num_classes):

        self.num_classes = num_classes

        self.confusion_matrix = np.zeros(
            (
                num_classes,
                num_classes
            ),
            dtype=np.int64
        )

    def update(
        self,
        outputs,
        targets
    ):

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        predictions = (
            predictions
            .detach()
            .cpu()
            .numpy()
            .reshape(-1)
        )

        targets = (
            targets
            .detach()
            .cpu()
            .numpy()
            .reshape(-1)
        )

        valid_pixels = (
            (targets >= 0)
            & (
                targets
                < self.num_classes
            )
        )

        indices = (
            self.num_classes
            * targets[valid_pixels]
            + predictions[valid_pixels]
        )

        batch_matrix = np.bincount(
            indices,
            minlength=(
                self.num_classes ** 2
            )
        ).reshape(
            self.num_classes,
            self.num_classes
        )

        self.confusion_matrix += (
            batch_matrix
        )

    def calculate(self):

        matrix = (
            self.confusion_matrix
            .astype(np.float64)
        )

        true_positive = np.diag(
            matrix
        )

        false_positive = (
            matrix.sum(axis=0)
            - true_positive
        )

        false_negative = (
            matrix.sum(axis=1)
            - true_positive
        )

        iou_denominator = (
            true_positive
            + false_positive
            + false_negative
        )

        precision_denominator = (
            true_positive
            + false_positive
        )

        recall_denominator = (
            true_positive
            + false_negative
        )

        iou = np.divide(
            true_positive,
            iou_denominator,
            out=np.full(
                true_positive.shape,
                np.nan,
                dtype=float
            ),
            where=(
                iou_denominator != 0
            )
        )

        precision = np.divide(
            true_positive,
            precision_denominator,
            out=np.full(
                true_positive.shape,
                np.nan,
                dtype=float
            ),
            where=(
                precision_denominator != 0
            )
        )

        recall = np.divide(
            true_positive,
            recall_denominator,
            out=np.full(
                true_positive.shape,
                np.nan,
                dtype=float
            ),
            where=(
                recall_denominator != 0
            )
        )

        return {
            "iou": iou,
            "precision": precision,
            "recall": recall,
            "mean_iou": np.nanmean(iou)
        }


# ============================================================
# TRAIN ONE EPOCH
# ============================================================

def train_one_epoch(
    model,
    loader
):

    model.train()

    total_loss = 0.0

    for batch_number, (
        images,
        masks
    ) in enumerate(
        loader,
        start=1
    ):

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        masks = masks.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=AMP_ENABLED
        ):

            outputs = model(images)

            loss = combined_loss(
                outputs,
                masks
            )

        scaler.scale(
            loss
        ).backward()

        scaler.step(
            optimizer
        )

        scaler.update()

        total_loss += loss.item()

        print(
            f"\rTraining batch "
            f"{batch_number}/"
            f"{len(loader)} "
            f"| Loss: "
            f"{loss.item():.4f}",
            end=""
        )

    print()

    average_loss = (
        total_loss / len(loader)
    )

    return average_loss


# ============================================================
# VALIDATE ONE EPOCH
# ============================================================

def validate_one_epoch(
    model,
    loader
):

    model.eval()

    total_loss = 0.0

    metrics = SegmentationMetrics(
        NUM_CLASSES
    )

    with torch.no_grad():

        for images, masks in loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            masks = masks.to(
                DEVICE,
                non_blocking=True
            )

            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=AMP_ENABLED
            ):

                outputs = model(images)

                loss = combined_loss(
                    outputs,
                    masks
                )

            total_loss += loss.item()

            metrics.update(
                outputs,
                masks
            )

    average_loss = (
        total_loss / len(loader)
    )

    metric_results = (
        metrics.calculate()
    )

    return (
        average_loss,
        metric_results
    )


print(
    "Training and validation functions ready"
)


# ============================================================
# TRAINING LOOP
# ============================================================

GROUND_WEEDS_INDEX = (
    class_names.index("Ground Weeds")
)

best_mean_iou = -1.0
best_ground_weeds_iou = -1.0

# This separate value is used only for early stopping.
# It prevents very tiny improvements from resetting patience.
best_ground_weeds_iou_for_stopping = -1.0

best_mean_iou_epoch = 0
best_ground_weeds_epoch = 0

BEST_MIOU_MODEL_PATH = (
    OUTPUT_DIR / "best_deeplabv3plus.pt"
)

BEST_GROUND_MODEL_PATH = (
    OUTPUT_DIR
    / "best_ground_weeds_deeplabv3plus.pt"
)

training_history = []

# Early stopping is based on validation Ground Weeds IoU.
epochs_without_improvement = 0

print("=" * 70)
print("Starting DeepLabV3+ training")
print("=" * 70)


for epoch in range(
    1,
    EPOCHS + 1
):

    print()
    print(f"Epoch {epoch}/{EPOCHS}")
    print("-" * 70)

    # ========================================================
    # TRAIN
    # ========================================================

    train_loss = train_one_epoch(
        model,
        train_loader
    )

    # ========================================================
    # VALIDATE
    # ========================================================

    (
        validation_loss,
        validation_metrics
    ) = validate_one_epoch(
        model,
        valid_loader
    )

    current_mean_iou = float(
        validation_metrics["mean_iou"]
    )

    current_ground_weeds_iou = float(
        validation_metrics["iou"][
            GROUND_WEEDS_INDEX
        ]
    )

    current_lr = (
        optimizer.param_groups[0]["lr"]
    )

    print(
        f"Training loss:   "
        f"{train_loss:.4f}"
    )

    print(
        f"Validation loss: "
        f"{validation_loss:.4f}"
    )

    print(
        f"Validation mIoU: "
        f"{current_mean_iou:.4f}"
    )

    print(
        f"Learning rate:   "
        f"{current_lr:.8f}"
    )

    epoch_record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "validation_loss": validation_loss,
        "mean_iou": current_mean_iou,
        "learning_rate": current_lr
    }

    # ========================================================
    # PER-CLASS METRICS
    # ========================================================

    for class_index, class_name in enumerate(
        class_names
    ):

        class_iou = float(
            validation_metrics["iou"][
                class_index
            ]
        )

        class_precision = float(
            validation_metrics["precision"][
                class_index
            ]
        )

        class_recall = float(
            validation_metrics["recall"][
                class_index
            ]
        )

        print(
            f"{class_name}: "
            f"IoU={class_iou:.4f} | "
            f"Precision={class_precision:.4f} | "
            f"Recall={class_recall:.4f}"
        )

        safe_class_name = (
            class_name
            .lower()
            .replace(" ", "_")
            .replace("/", "_")
        )

        epoch_record[
            f"{safe_class_name}_iou"
        ] = class_iou

        epoch_record[
            f"{safe_class_name}_precision"
        ] = class_precision

        epoch_record[
            f"{safe_class_name}_recall"
        ] = class_recall

    training_history.append(
        epoch_record
    )

    pd.DataFrame(
        training_history
    ).to_csv(
        HISTORY_PATH,
        index=False
    )

    # ========================================================
    # CHECKPOINT CONTENT
    # ========================================================

    checkpoint = {
        "epoch":
            epoch,

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scheduler_state_dict":
            scheduler.state_dict(),

        "validation_loss":
            validation_loss,

        "mean_iou":
            current_mean_iou,

        "ground_weeds_iou":
            current_ground_weeds_iou,

        "class_names":
            class_names,

        "pixel_to_class":
            pixel_to_class,

        "image_size":
            IMG_SIZE,

        "batch_size":
            BATCH_SIZE,

        "encoder":
            ENCODER,

        "encoder_weights":
            ENCODER_WEIGHTS,

        "learning_rate":
            LEARNING_RATE,

        "weight_decay":
            WEIGHT_DECAY
    }

    # ========================================================
    # SAVE BEST OVERALL MODEL
    # Highest validation mean IoU
    # ========================================================

    if current_mean_iou > best_mean_iou:

        best_mean_iou = (
            current_mean_iou
        )

        best_mean_iou_epoch = (
            epoch
        )

        torch.save(
            checkpoint,
            BEST_MIOU_MODEL_PATH
        )

        print(
            f"Saved BEST overall model "
            f"(mIoU={current_mean_iou:.4f})"
        )

    # ========================================================
    # SAVE BEST GROUND WEEDS MODEL
    # Highest validation Ground Weeds IoU
    # ========================================================

    if (
        current_ground_weeds_iou
        > best_ground_weeds_iou
    ):

        best_ground_weeds_iou = (
            current_ground_weeds_iou
        )

        best_ground_weeds_epoch = (
            epoch
        )

        torch.save(
            checkpoint,
            BEST_GROUND_MODEL_PATH
        )

        print(
            f"Saved BEST Ground Weeds model "
            f"(IoU={current_ground_weeds_iou:.4f})"
        )

    # ========================================================
    # EARLY STOPPING BASED ON GROUND WEEDS IoU
    # ========================================================

    meaningful_ground_weeds_improvement = (
        current_ground_weeds_iou
        > (
            best_ground_weeds_iou_for_stopping
            + MIN_DELTA
        )
    )

    if meaningful_ground_weeds_improvement:

        best_ground_weeds_iou_for_stopping = (
            current_ground_weeds_iou
        )

        epochs_without_improvement = 0

    else:

        epochs_without_improvement += 1

    print(
        f"Ground Weeds early-stopping counter: "
        f"{epochs_without_improvement}/"
        f"{PATIENCE}"
    )

    # ========================================================
    # SAVE LATEST MODEL EVERY EPOCH
    # ========================================================

    torch.save(
        checkpoint,
        LAST_MODEL_PATH
    )

    # ========================================================
    # UPDATE LEARNING RATE
    # ========================================================

    scheduler.step()

    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if (
        epochs_without_improvement
        >= PATIENCE
    ):

        print()
        print("=" * 70)
        print("EARLY STOPPING")
        print("=" * 70)

        print(
            f"No meaningful validation Ground Weeds "
            f"IoU improvement for {PATIENCE} epochs."
        )

        print()

        print(
            f"Best Ground Weeds epoch: "
            f"{best_ground_weeds_epoch}"
        )

        print(
            f"Best Ground Weeds IoU: "
            f"{best_ground_weeds_iou:.4f}"
        )

        print()

        print(
            f"Best overall epoch: "
            f"{best_mean_iou_epoch}"
        )

        print(
            f"Best overall mIoU: "
            f"{best_mean_iou:.4f}"
        )

        break


# ============================================================
# TRAINING SUMMARY
# ============================================================

print()
print("=" * 70)
print("TRAINING COMPLETED")
print("=" * 70)

print()
print("Best overall model:")

print(
    f"  Epoch: "
    f"{best_mean_iou_epoch}"
)

print(
    f"  Mean IoU: "
    f"{best_mean_iou:.4f}"
)

print(
    f"  Saved to: "
    f"{BEST_MIOU_MODEL_PATH.resolve()}"
)

print()
print("Best Ground Weeds model:")

print(
    f"  Epoch: "
    f"{best_ground_weeds_epoch}"
)

print(
    f"  Ground Weeds IoU: "
    f"{best_ground_weeds_iou:.4f}"
)

print(
    f"  Saved to: "
    f"{BEST_GROUND_MODEL_PATH.resolve()}"
)

print()

print(
    "Latest model saved to:",
    LAST_MODEL_PATH.resolve()
)

print(
    "Training history saved to:",
    HISTORY_PATH.resolve()
)

Random seed set to 67
Project root: /home/project/26060809/Blessco - DeepLabv3+/exp-6_real_aug10x-syn_aug5x
Dataset root: /home/project/26060809/Blessco - DeepLabv3+/exp-6_real_aug10x-syn_aug5x
Output folder: /home/project/26060809/Blessco - DeepLabv3+/exp-6_real_aug10x-syn_aug5x/deeplab_results
Device: cuda
GPU: NVIDIA L4


,Pixel Value,Class
0,0,background
1,1,Ground Weeds
2,2,Palm Oil Tree Trunk


Class names: ['background', 'Ground Weeds', 'Palm Oil Tree Trunk']
Number of classes: 3
Pixel mapping: {0: 0, 1: 1, 2: 2}
train: 2662 image-mask pairs
valid: 69 image-mask pairs
test: 35 image-mask pairs
syn_images: 243 synthetic samples
synaug_train_5x: 1458 synthetic samples

========== Dataset Summary ==========
Real Train          : 2662
Synthetic           : 243
Synthetic Augmented : 1458
Combined Train      : 4363
Validation          : 69
Test                : 35

========== DataLoaders ==========
Training batches   : 546
Validation batches : 9
Test batches       : 5
DeepLabV3+ model created
Encoder: resnet50
Number of output classes: 3
Model device: cuda:0
Training and validation functions ready
Starting DeepLabV3+ training

Epoch 1/50
----------------------------------------------------------------------
Training batch 546/546 | Loss: 0.2359
Training loss:   0.5477
Validation loss: 0.3090
Validation mIoU: 0.8369
Learning rate:   0.00010000
background: IoU=0.8702 | Precision=0.9

In [2]:
# EVALUATION MATRIX CODE
# ============================================================
# POST-TRAINING EVALUATION AND RESULT VISUALISATION
# ============================================================

EVALUATION_DIR = OUTPUT_DIR / "evaluation_results"

EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BEST_MIOU_EVALUATION_DIR = (
    EVALUATION_DIR / "best_miou_model"
)

BEST_GROUND_EVALUATION_DIR = (
    EVALUATION_DIR / "best_ground_weeds_model"
)

BEST_MIOU_EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BEST_GROUND_EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# LOAD A SAVED DEEPLABV3+ CHECKPOINT
# ============================================================

def load_deeplab_checkpoint(
    checkpoint_path
):

    checkpoint_path = Path(
        checkpoint_path
    )

    if not checkpoint_path.is_file():
        raise FileNotFoundError(
            f"Checkpoint not found:\n"
            f"{checkpoint_path.resolve()}"
        )

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False
    )

    if not isinstance(checkpoint, dict):
        raise TypeError(
            "Expected a checkpoint dictionary, "
            f"but received {type(checkpoint)}"
        )

    if "model_state_dict" not in checkpoint:
        raise KeyError(
            "Checkpoint does not contain "
            "'model_state_dict'.\n\n"
            f"Available keys: "
            f"{list(checkpoint.keys())}"
        )

    checkpoint_encoder = checkpoint.get(
        "encoder",
        ENCODER
    )

    checkpoint_class_names = checkpoint.get(
        "class_names",
        class_names
    )

    checkpoint_num_classes = len(
        checkpoint_class_names
    )

    loaded_model = smp.DeepLabV3Plus(
        encoder_name=checkpoint_encoder,
        encoder_weights=None,
        in_channels=3,
        classes=checkpoint_num_classes,
        activation=None
    )

    loaded_model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    loaded_model = loaded_model.to(
        DEVICE
    )

    loaded_model.eval()

    print()
    print("Loaded checkpoint:")
    print(checkpoint_path.resolve())
    print(
        "Epoch:",
        checkpoint.get("epoch", "Unknown")
    )
    print(
        "Encoder:",
        checkpoint_encoder
    )
    print(
        "Mean IoU:",
        checkpoint.get("mean_iou", "Unknown")
    )
    print(
        "Ground Weeds IoU:",
        checkpoint.get(
            "ground_weeds_iou",
            "Unknown"
        )
    )

    return (
        loaded_model,
        checkpoint,
        checkpoint_class_names
    )


# ============================================================
# EVALUATE MODEL ON ONE DATA LOADER
# ============================================================

def evaluate_complete(
    model,
    loader,
    split_name
):

    model.eval()

    total_loss = 0.0

    metric_tracker = SegmentationMetrics(
        NUM_CLASSES
    )

    with torch.no_grad():

        for batch_number, (
            images,
            masks
        ) in enumerate(
            loader,
            start=1
        ):

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            masks = masks.to(
                DEVICE,
                non_blocking=True
            )

            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=AMP_ENABLED
            ):

                outputs = model(images)

                loss = combined_loss(
                    outputs,
                    masks
                )

            total_loss += loss.item()

            metric_tracker.update(
                outputs,
                masks
            )

            print(
                f"\rEvaluating {split_name}: "
                f"batch {batch_number}/"
                f"{len(loader)}",
                end=""
            )

    print()

    average_loss = (
        total_loss / len(loader)
    )

    results = metric_tracker.calculate()

    results["loss"] = average_loss

    confusion_matrix = (
        metric_tracker.confusion_matrix.copy()
    )

    results["confusion_matrix"] = (
        confusion_matrix
    )

    overall_pixel_accuracy = (
        np.trace(confusion_matrix)
        / np.maximum(
            confusion_matrix.sum(),
            1
        )
    )

    results[
        "overall_pixel_accuracy"
    ] = overall_pixel_accuracy

    return results


# ============================================================
# CREATE PER-CLASS METRICS TABLE
# ============================================================

def create_metrics_dataframe(
    results,
    names
):

    rows = []

    confusion_matrix = (
        results["confusion_matrix"]
    )

    class_pixel_counts = (
        confusion_matrix.sum(axis=1)
    )

    for class_index, class_name in enumerate(
        names
    ):

        rows.append(
            {
                "Class": class_name,

                "IoU":
                    results["iou"][
                        class_index
                    ],

                "Precision":
                    results["precision"][
                        class_index
                    ],

                "Recall":
                    results["recall"][
                        class_index
                    ],

                "Ground Truth Pixels":
                    int(
                        class_pixel_counts[
                            class_index
                        ]
                    )
            }
        )

    dataframe = pd.DataFrame(
        rows
    )

    return dataframe


# ============================================================
# PRINT METRICS
# ============================================================

def print_evaluation_results(
    results,
    names,
    title
):

    print()
    print("=" * 70)
    print(title)
    print("=" * 70)

    print(
        f"Loss:                   "
        f"{results['loss']:.4f}"
    )

    print(
        f"Mean IoU:               "
        f"{results['mean_iou']:.4f}"
    )

    print(
        f"Overall Pixel Accuracy: "
        f"{results['overall_pixel_accuracy']:.4f}"
    )

    print()

    for class_index, class_name in enumerate(
        names
    ):

        print(
            f"{class_name}: "
            f"IoU={results['iou'][class_index]:.4f} | "
            f"Precision="
            f"{results['precision'][class_index]:.4f} | "
            f"Recall="
            f"{results['recall'][class_index]:.4f}"
        )


# ============================================================
# PLOT CONFUSION MATRIX
# ============================================================

def plot_confusion_matrix(
    confusion_matrix,
    names,
    title,
    save_path,
    normalize=False
):

    matrix = confusion_matrix.astype(
        np.float64
    )

    if normalize:

        row_totals = matrix.sum(
            axis=1,
            keepdims=True
        )

        matrix = np.divide(
            matrix,
            row_totals,
            out=np.zeros_like(
                matrix,
                dtype=np.float64
            ),
            where=row_totals != 0
        )

    figure, axis = plt.subplots(
        figsize=(9, 8)
    )

    image = axis.imshow(
        matrix
    )

    figure.colorbar(
        image,
        ax=axis
    )

    axis.set_xticks(
        np.arange(len(names))
    )

    axis.set_yticks(
        np.arange(len(names))
    )

    axis.set_xticklabels(
        names,
        rotation=30,
        ha="right"
    )

    axis.set_yticklabels(
        names
    )

    axis.set_xlabel(
        "Predicted class"
    )

    axis.set_ylabel(
        "Ground-truth class"
    )

    axis.set_title(
        title
    )

    threshold = (
        matrix.max() / 2
        if matrix.size > 0
        else 0
    )

    for row_index in range(
        matrix.shape[0]
    ):

        for column_index in range(
            matrix.shape[1]
        ):

            value = matrix[
                row_index,
                column_index
            ]

            if normalize:
                displayed_value = (
                    f"{value:.3f}"
                )
            else:
                displayed_value = (
                    f"{int(value):,}"
                )

            axis.text(
                column_index,
                row_index,
                displayed_value,
                ha="center",
                va="center"
            )

    figure.tight_layout()

    figure.savefig(
        save_path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.close(figure)


# ============================================================
# SAVE METRICS AS JSON
# ============================================================

def convert_results_for_json(
    results
):

    return {
        "loss": float(
            results["loss"]
        ),

        "mean_iou": float(
            results["mean_iou"]
        ),

        "overall_pixel_accuracy": float(
            results[
                "overall_pixel_accuracy"
            ]
        ),

        "iou": [
            float(value)
            for value in results["iou"]
        ],

        "precision": [
            float(value)
            for value in results["precision"]
        ],

        "recall": [
            float(value)
            for value in results["recall"]
        ],

        "confusion_matrix":
            results[
                "confusion_matrix"
            ].tolist()
    }


# ============================================================
# EVALUATE ONE CHECKPOINT
# ============================================================

def evaluate_checkpoint(
    checkpoint_path,
    model_name,
    evaluation_directory
):

    (
        evaluation_model,
        checkpoint,
        checkpoint_class_names
    ) = load_deeplab_checkpoint(
        checkpoint_path
    )

    if len(checkpoint_class_names) != NUM_CLASSES:
        raise ValueError(
            "Checkpoint number of classes does "
            "not match the current dataset.\n"
            f"Checkpoint classes: "
            f"{checkpoint_class_names}\n"
            f"Current classes: {class_names}"
        )

    # ------------------------
    # Validation evaluation
    # ------------------------

    validation_results = evaluate_complete(
        evaluation_model,
        valid_loader,
        split_name="validation"
    )

    print_evaluation_results(
        validation_results,
        checkpoint_class_names,
        title=(
            f"{model_name} — "
            f"Validation Results"
        )
    )

    validation_dataframe = (
        create_metrics_dataframe(
            validation_results,
            checkpoint_class_names
        )
    )

    validation_dataframe.to_csv(
        evaluation_directory
        / "validation_metrics.csv",
        index=False
    )

    plot_confusion_matrix(
        validation_results[
            "confusion_matrix"
        ],
        checkpoint_class_names,
        title=(
            f"{model_name} — "
            f"Validation Confusion Matrix"
        ),
        save_path=(
            evaluation_directory
            / "validation_confusion_matrix.png"
        ),
        normalize=False
    )

    plot_confusion_matrix(
        validation_results[
            "confusion_matrix"
        ],
        checkpoint_class_names,
        title=(
            f"{model_name} — "
            f"Normalized Validation "
            f"Confusion Matrix"
        ),
        save_path=(
            evaluation_directory
            / (
                "validation_confusion_"
                "matrix_normalized.png"
            )
        ),
        normalize=True
    )

    # ------------------------
    # Test evaluation
    # ------------------------

    test_results = evaluate_complete(
        evaluation_model,
        test_loader,
        split_name="test"
    )

    print_evaluation_results(
        test_results,
        checkpoint_class_names,
        title=(
            f"{model_name} — "
            f"Test Results"
        )
    )

    test_dataframe = (
        create_metrics_dataframe(
            test_results,
            checkpoint_class_names
        )
    )

    test_dataframe.to_csv(
        evaluation_directory
        / "test_metrics.csv",
        index=False
    )

    plot_confusion_matrix(
        test_results[
            "confusion_matrix"
        ],
        checkpoint_class_names,
        title=(
            f"{model_name} — "
            f"Test Confusion Matrix"
        ),
        save_path=(
            evaluation_directory
            / "test_confusion_matrix.png"
        ),
        normalize=False
    )

    plot_confusion_matrix(
        test_results[
            "confusion_matrix"
        ],
        checkpoint_class_names,
        title=(
            f"{model_name} — "
            f"Normalized Test "
            f"Confusion Matrix"
        ),
        save_path=(
            evaluation_directory
            / (
                "test_confusion_"
                "matrix_normalized.png"
            )
        ),
        normalize=True
    )

    summary = {
        "checkpoint": str(
            Path(checkpoint_path).resolve()
        ),

        "saved_epoch": checkpoint.get(
            "epoch"
        ),

        "encoder": checkpoint.get(
            "encoder"
        ),

        "checkpoint_validation_miou":
            checkpoint.get(
                "mean_iou"
            ),

        "checkpoint_validation_ground_weeds_iou":
            checkpoint.get(
                "ground_weeds_iou"
            ),

        "validation":
            convert_results_for_json(
                validation_results
            ),

        "test":
            convert_results_for_json(
                test_results
            )
    }

    with open(
        evaluation_directory
        / "evaluation_summary.json",
        "w",
        encoding="utf-8"
    ) as json_file:

        json.dump(
            summary,
            json_file,
            indent=4
        )

    return {
        "validation":
            validation_results,

        "test":
            test_results,

        "checkpoint":
            checkpoint
    }


# ============================================================
# TRAINING HISTORY PLOTS
# ============================================================

def save_training_history_plots(
    history_path,
    output_directory
):

    history = pd.read_csv(
        history_path
    )

    required_columns = [
        "epoch",
        "train_loss",
        "validation_loss",
        "mean_iou",
        "ground_weeds_iou",
        "ground_weeds_precision",
        "ground_weeds_recall"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in history.columns
    ]

    if missing_columns:
        raise KeyError(
            f"Missing history columns: "
            f"{missing_columns}\n\n"
            f"Available columns:\n"
            f"{history.columns.tolist()}"
        )

    # ------------------------
    # Loss graph
    # ------------------------

    figure, axis = plt.subplots(
        figsize=(10, 6)
    )

    axis.plot(
        history["epoch"],
        history["train_loss"],
        label="Training loss"
    )

    axis.plot(
        history["epoch"],
        history["validation_loss"],
        label="Validation loss"
    )

    axis.set_xlabel("Epoch")
    axis.set_ylabel("Loss")
    axis.set_title(
        "DeepLabV3+ Training and Validation Loss"
    )
    axis.grid(True)
    axis.legend()

    figure.tight_layout()

    figure.savefig(
        output_directory
        / "loss_curves.png",
        dpi=200,
        bbox_inches="tight"
    )

    plt.close(figure)

    # ------------------------
    # Mean IoU graph
    # ------------------------

    figure, axis = plt.subplots(
        figsize=(10, 6)
    )

    axis.plot(
        history["epoch"],
        history["mean_iou"]
    )

    axis.set_xlabel("Epoch")
    axis.set_ylabel("Mean IoU")
    axis.set_title(
        "DeepLabV3+ Validation Mean IoU"
    )
    axis.grid(True)

    figure.tight_layout()

    figure.savefig(
        output_directory
        / "mean_iou_curve.png",
        dpi=200,
        bbox_inches="tight"
    )

    plt.close(figure)

    # ------------------------
    # Ground Weeds metrics graph
    # ------------------------

    figure, axis = plt.subplots(
        figsize=(10, 6)
    )

    axis.plot(
        history["epoch"],
        history["ground_weeds_iou"],
        label="Ground Weeds IoU"
    )

    axis.plot(
        history["epoch"],
        history[
            "ground_weeds_precision"
        ],
        label="Ground Weeds precision"
    )

    axis.plot(
        history["epoch"],
        history[
            "ground_weeds_recall"
        ],
        label="Ground Weeds recall"
    )

    axis.set_xlabel("Epoch")
    axis.set_ylabel("Metric")
    axis.set_title(
        "Ground Weeds Validation Metrics"
    )
    axis.grid(True)
    axis.legend()

    figure.tight_layout()

    figure.savefig(
        output_directory
        / "ground_weeds_metrics.png",
        dpi=200,
        bbox_inches="tight"
    )

    plt.close(figure)


# ============================================================
# YOLO-STYLE COMBINED RESULTS GRAPH
# ============================================================

def save_combined_results_plot(
    history_path,
    save_path
):

    history = pd.read_csv(
        history_path
    )

    figure = plt.figure(
        figsize=(16, 12)
    )

    # Training loss
    axis_1 = figure.add_subplot(
        2,
        2,
        1
    )

    axis_1.plot(
        history["epoch"],
        history["train_loss"],
        label="Training loss"
    )

    axis_1.plot(
        history["epoch"],
        history["validation_loss"],
        label="Validation loss"
    )

    axis_1.set_title("Loss")
    axis_1.set_xlabel("Epoch")
    axis_1.set_ylabel("Loss")
    axis_1.grid(True)
    axis_1.legend()

    # Mean IoU
    axis_2 = figure.add_subplot(
        2,
        2,
        2
    )

    axis_2.plot(
        history["epoch"],
        history["mean_iou"]
    )

    axis_2.set_title(
        "Validation Mean IoU"
    )
    axis_2.set_xlabel("Epoch")
    axis_2.set_ylabel("Mean IoU")
    axis_2.grid(True)

    # Ground Weeds IoU
    axis_3 = figure.add_subplot(
        2,
        2,
        3
    )

    axis_3.plot(
        history["epoch"],
        history["ground_weeds_iou"]
    )

    axis_3.set_title(
        "Ground Weeds IoU"
    )
    axis_3.set_xlabel("Epoch")
    axis_3.set_ylabel("IoU")
    axis_3.grid(True)

    # Ground Weeds precision and recall
    axis_4 = figure.add_subplot(
        2,
        2,
        4
    )

    axis_4.plot(
        history["epoch"],
        history[
            "ground_weeds_precision"
        ],
        label="Precision"
    )

    axis_4.plot(
        history["epoch"],
        history[
            "ground_weeds_recall"
        ],
        label="Recall"
    )

    axis_4.set_title(
        "Ground Weeds Precision and Recall"
    )
    axis_4.set_xlabel("Epoch")
    axis_4.set_ylabel("Metric")
    axis_4.grid(True)
    axis_4.legend()

    figure.suptitle(
        "DeepLabV3+ Training Results",
        fontsize=16
    )

    figure.tight_layout()

    figure.savefig(
        save_path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.close(figure)


# ============================================================
# GENERATE TRAINING GRAPHS
# ============================================================

print()
print("Generating training graphs...")

save_training_history_plots(
    HISTORY_PATH,
    EVALUATION_DIR
)

save_combined_results_plot(
    HISTORY_PATH,
    EVALUATION_DIR / "results.png"
)


# ============================================================
# EVALUATE BEST OVERALL MODEL
# ============================================================

print()
print("=" * 70)
print("EVALUATING BEST OVERALL MODEL")
print("=" * 70)

best_miou_results = evaluate_checkpoint(
    checkpoint_path=BEST_MIOU_MODEL_PATH,
    model_name="Best Mean IoU Model",
    evaluation_directory=(
        BEST_MIOU_EVALUATION_DIR
    )
)


# ============================================================
# EVALUATE BEST GROUND WEEDS MODEL
# ============================================================

print()
print("=" * 70)
print("EVALUATING BEST GROUND WEEDS MODEL")
print("=" * 70)

best_ground_results = evaluate_checkpoint(
    checkpoint_path=BEST_GROUND_MODEL_PATH,
    model_name="Best Ground Weeds Model",
    evaluation_directory=(
        BEST_GROUND_EVALUATION_DIR
    )
)


# ============================================================
# COMPARISON TABLE
# ============================================================

comparison_rows = []

for model_label, result_bundle in [
    (
        "Best Mean IoU Model",
        best_miou_results
    ),
    (
        "Best Ground Weeds Model",
        best_ground_results
    )
]:

    validation_result = (
        result_bundle["validation"]
    )

    test_result = (
        result_bundle["test"]
    )

    comparison_rows.append(
        {
            "Model": model_label,

            "Saved Epoch":
                result_bundle[
                    "checkpoint"
                ].get("epoch"),

            "Validation mIoU":
                validation_result[
                    "mean_iou"
                ],

            "Validation Pixel Accuracy":
                validation_result[
                    "overall_pixel_accuracy"
                ],

            "Validation Ground Weeds IoU":
                validation_result[
                    "iou"
                ][GROUND_WEEDS_INDEX],

            "Validation Ground Weeds Precision":
                validation_result[
                    "precision"
                ][GROUND_WEEDS_INDEX],

            "Validation Ground Weeds Recall":
                validation_result[
                    "recall"
                ][GROUND_WEEDS_INDEX],

            "Test mIoU":
                test_result[
                    "mean_iou"
                ],

            "Test Pixel Accuracy":
                test_result[
                    "overall_pixel_accuracy"
                ],

            "Test Ground Weeds IoU":
                test_result[
                    "iou"
                ][GROUND_WEEDS_INDEX],

            "Test Ground Weeds Precision":
                test_result[
                    "precision"
                ][GROUND_WEEDS_INDEX],

            "Test Ground Weeds Recall":
                test_result[
                    "recall"
                ][GROUND_WEEDS_INDEX]
        }
    )


comparison_dataframe = pd.DataFrame(
    comparison_rows
)

comparison_path = (
    EVALUATION_DIR
    / "checkpoint_comparison.csv"
)

comparison_dataframe.to_csv(
    comparison_path,
    index=False
)

print()
print("=" * 70)
print("CHECKPOINT COMPARISON")
print("=" * 70)

display(
    comparison_dataframe
)

print()
print("All evaluation results saved to:")
print(EVALUATION_DIR.resolve())


Generating training graphs...

EVALUATING BEST OVERALL MODEL

Loaded checkpoint:
/home/project/26060809/Blessco - DeepLabv3+/exp-6_real_aug10x-syn_aug5x/deeplab_results/best_deeplabv3plus.pt
Epoch: 30
Encoder: resnet50
Mean IoU: 0.8457744881341333
Ground Weeds IoU: 0.7398161305823749
Evaluating validation: batch 9/9

Best Mean IoU Model — Validation Results
Loss:                   0.5198
Mean IoU:               0.8458
Overall Pixel Accuracy: 0.9377

background: IoU=0.8726 | Precision=0.9199 | Recall=0.9444
Ground Weeds: IoU=0.7398 | Precision=0.8874 | Recall=0.8165
Palm Oil Tree Trunk: IoU=0.9249 | Precision=0.9651 | Recall=0.9568
Evaluating test: batch 5/5

Best Mean IoU Model — Test Results
Loss:                   0.4928
Mean IoU:               0.8257
Overall Pixel Accuracy: 0.9391

background: IoU=0.8818 | Precision=0.9235 | Recall=0.9513
Ground Weeds: IoU=0.6643 | Precision=0.8666 | Recall=0.7400
Palm Oil Tree Trunk: IoU=0.9310 | Precision=0.9665 | Recall=0.9621

EVALUATING BEST G

,Model,Saved Epoch,Validation mIoU,Validation Pixel Accuracy,Validation Ground Weeds IoU,Validation Ground Weeds Precision,Validation Ground Weeds Recall,Test mIoU,Test Pixel Accuracy,Test Ground Weeds IoU,Test Ground Weeds Precision,Test Ground Weeds Recall
0,Best Mean IoU Model,30,0.845774,0.937702,0.739816,0.887399,0.816461,0.825707,0.939137,0.664318,0.866551,0.740027
1,Best Ground Weeds Model,30,0.845774,0.937702,0.739816,0.887399,0.816461,0.825707,0.939137,0.664318,0.866551,0.740027



All evaluation results saved to:
/home/project/26060809/Blessco - DeepLabv3+/exp-6_real_aug10x-syn_aug5x/deeplab_results/evaluation_results


In [3]:
# VISUALIZATION CODE
# ============================================================
# DEEPLABV3+ TEST PREDICTION VISUALIZATION
# Uses best_deeplabv3plus.pt
# MUST USE THE SAME MODEL CONFIGURATIONS
# ============================================================

from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

import torch
import segmentation_models_pytorch as smp

import albumentations as A
from albumentations.pytorch import ToTensorV2


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_ROOT = Path.cwd()

TEST_DIR = PROJECT_ROOT / "test"

# Images and masks are stored together
TEST_IMAGES_DIR = TEST_DIR
TEST_MASKS_DIR = TEST_DIR

OUTPUT_DIR = PROJECT_ROOT / "deeplab_results"

BEST_MODEL_PATH = (
    OUTPUT_DIR / "best_ground_weeds_deeplabv3plus.pt"
)

VISUALIZATION_DIR = (
    OUTPUT_DIR / "test_visualizations"
)

RAW_PREDICTION_DIR = (
    OUTPUT_DIR / "test_predicted_masks"
)

VISUALIZATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RAW_PREDICTION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# 2. MODEL SETTINGS
# These must match the training configuration
# ============================================================

IMG_SIZE = 1024

ENCODER = "resnet50"
ENCODER_WEIGHTS = "imagenet"

NUM_CLASSES = 3

CLASS_NAMES = {
    0: "background",
    1: "ground_weeds",
    2: "trunk_and_epiphytes",
}


# ============================================================
# 3. DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)
print("Best model:", BEST_MODEL_PATH)

if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Could not find best model:\n"
        f"{BEST_MODEL_PATH}"
    )

if not TEST_IMAGES_DIR.exists():
    raise FileNotFoundError(
        f"Could not find test images folder:\n"
        f"{TEST_IMAGES_DIR}"
    )

if not TEST_MASKS_DIR.exists():
    raise FileNotFoundError(
        f"Could not find test masks folder:\n"
        f"{TEST_MASKS_DIR}"
    )


# ============================================================
# 4. CREATE MODEL
# ============================================================

model = smp.DeepLabV3Plus(
    encoder_name=ENCODER,
    encoder_weights=None,
    in_channels=3,
    classes=NUM_CLASSES,
)

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
)

# Supports either a plain state_dict or a saved checkpoint dictionary
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    model.load_state_dict(
        checkpoint["state_dict"]
    )

else:
    model.load_state_dict(checkpoint)

model.to(DEVICE)
model.eval()

print("[OK] Best DeepLabV3+ model loaded.")


# ============================================================
# 5. INFERENCE TRANSFORM
# Must match the validation/test preprocessing used in training
# ============================================================

inference_transform = A.Compose([
    A.Resize(
        height=IMG_SIZE,
        width=IMG_SIZE,
    ),
    A.Normalize(),
    ToTensorV2(),
])


# ============================================================
# FIND TEST IMAGES
# ============================================================

image_paths = sorted(
    path
    for path in TEST_DIR.iterdir()
    if (
        path.is_file()
        and path.suffix.lower() in {".jpg", ".jpeg"}
        and not path.stem.endswith("_mask")
    )
)

if not image_paths:
    raise RuntimeError(
        f"No test images found in:\n{TEST_DIR}"
    )

print(f"Test images found: {len(image_paths)}")

# ============================================================
# 7. MASK PATH HELPER
# ============================================================

def find_mask_path(image_path: Path):
    """
    Example:

    image:
    A12_jpg.rf.xxxxx.jpg

    mask:
    A12_jpg.rf.xxxxx_mask.png
    """

    mask_path = TEST_DIR / f"{image_path.stem}_mask.png"

    if mask_path.exists():
        return mask_path

    return None


# ============================================================
# 8. COLOUR HELPERS
# ============================================================

# RGB colours used only for visualisation
CLASS_COLOURS = np.array([
    [0, 0, 0],        # background
    [0, 255, 0],      # ground weeds
    [255, 165, 0],    # trunk and epiphytes
], dtype=np.uint8)


def colourise_mask(mask):
    """
    Convert class-index mask into an RGB image.
    """

    mask = np.asarray(mask, dtype=np.int64)

    colour_mask = np.zeros(
        (*mask.shape, 3),
        dtype=np.uint8,
    )

    valid = (
        (mask >= 0)
        & (mask < len(CLASS_COLOURS))
    )

    colour_mask[valid] = (
        CLASS_COLOURS[mask[valid]]
    )

    return colour_mask


def create_overlay(
    image_rgb,
    colour_mask,
    alpha=0.45,
):
    """
    Overlay a coloured prediction mask on the image.
    """

    image_float = image_rgb.astype(
        np.float32
    )

    mask_float = colour_mask.astype(
        np.float32
    )

    foreground = np.any(
        colour_mask != 0,
        axis=2,
    )

    overlay = image_float.copy()

    overlay[foreground] = (
        (1.0 - alpha)
        * image_float[foreground]
        + alpha
        * mask_float[foreground]
    )

    return np.clip(
        overlay,
        0,
        255,
    ).astype(np.uint8)


# ============================================================
# 9. RUN INFERENCE AND SAVE RESULTS
# ============================================================

processed_count = 0
missing_mask_count = 0

with torch.no_grad():

    for index, image_path in enumerate(
        image_paths,
        start=1,
    ):

        mask_path = find_mask_path(image_path)

        if mask_path is None:
            print(
                f"[SKIPPED] No mask found for: "
                f"{image_path.name}"
            )

            missing_mask_count += 1
            continue

        # Read original image
        image_bgr = cv2.imread(
            str(image_path),
            cv2.IMREAD_COLOR,
        )

        if image_bgr is None:
            print(
                f"[SKIPPED] Could not read: "
                f"{image_path}"
            )
            continue

        image_rgb = cv2.cvtColor(
            image_bgr,
            cv2.COLOR_BGR2RGB,
        )

        original_height, original_width = (
            image_rgb.shape[:2]
        )

        # Read ground-truth mask
        ground_truth = cv2.imread(
            str(mask_path),
            cv2.IMREAD_GRAYSCALE,
        )

        if ground_truth is None:
            print(
                f"[SKIPPED] Could not read mask: "
                f"{mask_path}"
            )
            continue

        # Prepare model input
        transformed = inference_transform(
            image=image_rgb
        )

        image_tensor = (
            transformed["image"]
            .unsqueeze(0)
            .to(DEVICE)
        )

        # Predict
        logits = model(image_tensor)

        prediction = torch.argmax(
            logits,
            dim=1,
        )[0].cpu().numpy().astype(
            np.uint8
        )

        # Resize prediction back to original image size
        prediction_original_size = cv2.resize(
            prediction,
            (original_width, original_height),
            interpolation=cv2.INTER_NEAREST,
        )

        ground_truth_original_size = cv2.resize(
            ground_truth,
            (original_width, original_height),
            interpolation=cv2.INTER_NEAREST,
        )

        predicted_colour = colourise_mask(
            prediction_original_size
        )

        ground_truth_colour = colourise_mask(
            ground_truth_original_size
        )

        prediction_overlay = create_overlay(
            image_rgb,
            predicted_colour,
            alpha=0.45,
        )

        # Save raw class-index mask
        raw_prediction_path = (
            RAW_PREDICTION_DIR
            / f"{image_path.stem}_prediction.png"
        )

        cv2.imwrite(
            str(raw_prediction_path),
            prediction_original_size,
        )

        # Create four-panel figure
        figure, axes = plt.subplots(
            1,
            4,
            figsize=(24, 7),
        )

        axes[0].imshow(image_rgb)
        axes[0].set_title("Original image")
        axes[0].axis("off")

        axes[1].imshow(ground_truth_colour)
        axes[1].set_title("Ground-truth mask")
        axes[1].axis("off")

        axes[2].imshow(predicted_colour)
        axes[2].set_title("Predicted mask")
        axes[2].axis("off")

        axes[3].imshow(prediction_overlay)
        axes[3].set_title("Prediction overlay")
        axes[3].axis("off")

        figure.suptitle(
            image_path.name,
            fontsize=14,
        )

        figure.tight_layout()

        visualization_path = (
            VISUALIZATION_DIR
            / f"{image_path.stem}_comparison.png"
        )

        figure.savefig(
            visualization_path,
            dpi=180,
            bbox_inches="tight",
        )

        plt.close(figure)

        processed_count += 1

        print(
            f"[{index}/{len(image_paths)}] "
            f"Saved: {visualization_path.name}"
        )


# ============================================================
# 10. SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("DEEPLABV3+ VISUALISATION COMPLETED")
print("=" * 70)

print(f"Images processed:       {processed_count}")
print(f"Missing masks:          {missing_mask_count}")

print(
    f"\nComparison images saved to:\n"
    f"{VISUALIZATION_DIR}"
)

print(
    f"\nRaw predicted masks saved to:\n"
    f"{RAW_PREDICTION_DIR}"
)

Device: cuda
Best model: /home/project/26060809/Blessco - DeepLabv3+/exp-6_real_aug10x-syn_aug5x/deeplab_results/best_ground_weeds_deeplabv3plus.pt
[OK] Best DeepLabV3+ model loaded.
Test images found: 35
[1/35] Saved: A12_jpg.rf.4bf59903d4420e6f3afebd2e28e54c64_comparison.png
[2/35] Saved: A28_jpg.rf.0dfda88ffc303533230e1a490acc31b1_comparison.png
[3/35] Saved: A2_jpg.rf.f1a4e9123d8b53ad83ce6222330bf190_comparison.png
[4/35] Saved: A31_jpg.rf.f79bc9751de7f4d3e9029cfaab18f066_comparison.png
[5/35] Saved: A34_jpg.rf.8b4acac219c88fc1ce8a2e9548d5e693_comparison.png
[6/35] Saved: A46_jpg.rf.58caf0dfdc62ee16f0361786250ae37f_comparison.png
[7/35] Saved: A57_jpg.rf.6fff0a55aa147482bd363d86088a9da5_comparison.png
[8/35] Saved: A68_jpg.rf.d292badb63e038bdfed3fc8b09ca0c56_comparison.png
[9/35] Saved: A7_jpg.rf.bbb14badff539ccc49bf964bed33bf8f_comparison.png
[10/35] Saved: B10_jpg.rf.976d25e0becb88db46ad124810f47d27_comparison.png
[11/35] Saved: B18_jpg.rf.2e8f941a65033807d28ebaa45ffb392b_compari